In [1]:
from pathlib import Path
import json
import boto3

PROJECT_ROOT = Path(
    "/home/sagemaker-user/AAI-540-Project/"
    "multilingual-it-ticket-routing-mlops"
)

config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
sagemaker_client = aws_session.client("sagemaker")
s3_client = aws_session.client("s3")

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=project_config["deployment"]["job_name"]
)
assert batch_details["TransformJobStatus"] == "Completed"

deployed_model_name = batch_details["ModelName"]
deployed_model = sagemaker_client.describe_model(
    ModelName=deployed_model_name
)
serving_container = deployed_model["PrimaryContainer"]

print("Model used for batch predictions:", deployed_model_name)
print("Serving image:", serving_container["Image"])
print("Serving model artifact:", serving_container.get("ModelDataUrl"))
print(
    "Serving configuration keys:",
    sorted(serving_container.get("Environment", {})),
)

Model used for batch predictions: aai540-group3-routing-ce725539a788
Serving image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3
Serving model artifact: s3://aai540-group3-tickets-390568313988-us-east-1/training-output/aai540-group3-lr-20261004-003353/output/model.tar.gz
Serving configuration keys: ['SAGEMAKER_CONTAINER_LOG_LEVEL', 'SAGEMAKER_MODEL_SERVER_WORKERS', 'SAGEMAKER_PROGRAM', 'SAGEMAKER_REGION', 'SAGEMAKER_SUBMIT_DIRECTORY']



In [ ]:
import copy
import hashlib
import tarfile
from io import BytesIO
from urllib.parse import urlparse


def require_match(actual, expected, label):
    if actual != expected:
        raise ValueError(f"{label} does not match the saved registration.")


def read_json_object(path):
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path.name}")
    return value


# Reload the saved settings before checking the registry files.
project_config = read_json_object(config_path)
registry_checkpoint = copy.deepcopy(project_config.get("model_registry", {}))
if not isinstance(registry_checkpoint, dict):
    raise ValueError("The saved registry settings must be a JSON object.")

registry_report_dir = PROJECT_ROOT / "reports" / "model_registry"
assets_path = registry_report_dir / "verified_assets.json"
evaluation_path = registry_report_dir / "evaluation.json"
request_path = registry_report_dir / "registration_request.json"
summary_path = registry_report_dir / "registration_summary.json"
registered_resume = bool(registry_checkpoint.get("model_package_arn"))
completed_resume = registered_resume and registry_checkpoint.get("status") == "Completed"

if registry_checkpoint:
    for key in ("verified_assets", "evaluation_report"):
        if not isinstance(registry_checkpoint.get(key), dict):
            raise ValueError(f"Missing saved registry details: {key}")
    for path in (assets_path, evaluation_path):
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved registry file: {path.name}")
elif any(path.exists() for path in (
    assets_path, evaluation_path, request_path, summary_path
)):
    raise ValueError("Registry files exist without their saved configuration.")

if registered_resume:
    for key in ("model_package_group_name", "registration_client_token"):
        if not registry_checkpoint.get(key):
            raise ValueError(f"Missing saved registry details: {key}")
    if not request_path.is_file():
        raise FileNotFoundError("The saved registration request is missing.")
    if completed_resume and not summary_path.is_file():
        raise FileNotFoundError("The saved registration summary is missing.")
elif summary_path.exists():
    raise ValueError("A registration summary exists without a saved model ARN.")

if registry_checkpoint.get("registration_client_token") and not request_path.is_file():
    raise FileNotFoundError("The saved registration request is missing.")


def current_config():
    current = read_json_object(config_path)
    require_match(
        current.get("model_registry", {}), registry_checkpoint,
        "Registry settings changed while these cells were running",
    )
    return current


def save_registry_settings(updates):
    global project_config, registry_checkpoint
    current = current_config()
    updated = copy.deepcopy(registry_checkpoint)
    updated.update(copy.deepcopy(updates))
    current["model_registry"] = updated
    config_path.write_text(
        json.dumps(current, indent=2, allow_nan=False), encoding="utf-8"
    )
    project_config = current
    registry_checkpoint = copy.deepcopy(updated)


def save_new_json(path, value):
    if path.exists():
        require_match(read_json_object(path), value, path.name)
        return
    with path.open("x", encoding="utf-8") as file:
        json.dump(value, file, indent=2, allow_nan=False)


def read_s3_artifact(uri, version_id=None):
    location = urlparse(uri)
    if location.scheme != "s3" or not location.netloc or not location.path.lstrip("/"):
        raise ValueError(f"Expected an S3 file location: {uri}")
    request = {"Bucket": location.netloc, "Key": location.path.lstrip("/")}
    if version_id is not None:
        if not isinstance(version_id, str) or not version_id or version_id == "null":
            raise ValueError("A saved S3 version ID is missing.")
        request["VersionId"] = version_id
    response = s3_client.get_object(**request)
    with response["Body"] as body:
        content = body.read()
    actual_version = response.get("VersionId")
    if not actual_version or actual_version == "null":
        raise ValueError("The S3 object has no version ID.")
    if version_id is not None:
        require_match(actual_version, version_id, "S3 version")
    return content, {
        "uri": uri,
        "version_id": actual_version,
        "sha256": hashlib.sha256(content).hexdigest(),
    }


def check_current_s3_version(artifact):
    # Serving and registry report URLs do not include a version ID.
    location = urlparse(artifact["uri"])
    response = s3_client.head_object(
        Bucket=location.netloc, Key=location.path.lstrip("/")
    )
    require_match(
        response.get("VersionId"), artifact["version_id"],
        f"Current S3 version for {artifact['uri']}",
    )


final_model = project_config["final_model"]
require_match(
    serving_container["ModelDataUrl"], final_model["model_uri"], "Model location"
)
model_bytes, model_artifact = read_s3_artifact(
    final_model["model_uri"], final_model["model_version_id"]
)
require_match(
    model_artifact["sha256"], final_model["model_archive_sha256"], "Model SHA-256"
)
check_current_s3_version(model_artifact)

serving_environment = dict(serving_container["Environment"])
code_uri = serving_environment["SAGEMAKER_SUBMIT_DIRECTORY"]
saved_assets = registry_checkpoint.get("verified_assets", {})
saved_code = saved_assets.get("inference_code_artifact", {})
if saved_assets:
    for key in ("uri", "version_id", "sha256"):
        if not saved_code.get(key):
            raise ValueError(f"Missing inference code detail: {key}")
    require_match(code_uri, saved_code["uri"], "Inference code location")
code_bytes, code_artifact = read_s3_artifact(code_uri, saved_code.get("version_id"))
check_current_s3_version(code_artifact)
expected_source_hashes = project_config["deployment"]["inference_source_sha256"]

with tarfile.open(fileobj=BytesIO(code_bytes), mode="r:gz") as archive:
    for filename, expected_hash in expected_source_hashes.items():
        matches = [
            member for member in archive.getmembers()
            if member.isfile() and Path(member.name).name == filename
        ]
        if len(matches) != 1:
            raise ValueError(f"Missing or ambiguous file: {filename}")
        extracted = archive.extractfile(matches[0])
        if extracted is None:
            raise ValueError(f"Could not read inference file: {filename}")
        with extracted:
            actual_hash = hashlib.sha256(extracted.read()).hexdigest()
        require_match(actual_hash, expected_hash, f"Inference file {filename}")

registry_assets = {
    "source_model_name": deployed_model_name,
    "serving_image": serving_container["Image"],
    "serving_environment": serving_environment,
    "model_artifact": model_artifact,
    "inference_code_artifact": code_artifact,
    "inference_source_sha256": expected_source_hashes,
}
if registry_checkpoint:
    require_match(registry_assets, saved_assets, "Verified model and code")
if assets_path.exists():
    require_match(read_json_object(assets_path), registry_assets, assets_path.name)

print("Model and prediction code checked.")


In [ ]:
import math

test_metrics = project_config["final_model"]["test_metrics"]

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    value = float(test_metrics[metric])
    assert math.isfinite(value) and 0.0 <= value <= 1.0

registry_evaluation = {
    "evaluation_split": "test",
    "evaluated_tickets": project_config["final_model"]["test_rows"],
    "training_job_name": project_config["final_model"]["training_job_name"],
    "model_selection_metric": "validation_macro_f1",
    "evaluation_dataset": project_config["final_model"]["test_dataset"],
    "multiclass_classification_metrics": {
        metric: {"value": float(test_metrics[metric])}
        for metric in ["accuracy", "macro_f1", "weighted_f1"]
    },
    "limitations": [
        "Current accuracy is insufficient to establish readiness for "
        "fully automatic ticket routing.",
        "Performance differs across queues and languages.",
        "Small test samples limit conclusions for Spanish, French, "
        "and Portuguese.",
    ],
}

evaluation_bytes = json.dumps(
    registry_evaluation, indent=2, allow_nan=False
).encode("utf-8")

evaluation_sha256 = hashlib.sha256(evaluation_bytes).hexdigest()
evaluation_key = (
    f"registry/evaluations/{evaluation_sha256}/evaluation.json"
)

evaluation_uri = f"s3://{PROJECT_BUCKET}/{evaluation_key}"
if evaluation_path.exists():
    require_match(evaluation_path.read_bytes(), evaluation_bytes, evaluation_path.name)

if registry_checkpoint:
    evaluation_artifact = copy.deepcopy(registry_checkpoint["evaluation_report"])
    for key in ("uri", "version_id", "sha256"):
        if not evaluation_artifact.get(key):
            raise ValueError(f"Missing evaluation report detail: {key}")
    require_match(evaluation_artifact["uri"], evaluation_uri, "Evaluation location")
    require_match(evaluation_artifact["sha256"], evaluation_sha256, "Evaluation SHA-256")
    stored_bytes, stored_artifact = read_s3_artifact(
        evaluation_artifact["uri"], evaluation_artifact["version_id"]
    )
    require_match(stored_artifact, evaluation_artifact, "Saved evaluation report")
    require_match(stored_bytes, evaluation_bytes, "Evaluation contents")
    check_current_s3_version(evaluation_artifact)
    print("Using the saved evaluation report.")
else:
    current_config()
    uploaded = s3_client.put_object(
        Bucket=PROJECT_BUCKET,
        Key=evaluation_key,
        Body=evaluation_bytes,
        ContentType="application/json",
    )
    evaluation_version = uploaded.get("VersionId")
    if not evaluation_version or evaluation_version == "null":
        raise ValueError("The evaluation upload has no S3 version ID.")
    stored_bytes, evaluation_artifact = read_s3_artifact(
        evaluation_uri, evaluation_version
    )
    require_match(stored_bytes, evaluation_bytes, "Uploaded evaluation contents")
    registry_report_dir.mkdir(parents=True, exist_ok=True)
    save_new_json(assets_path, registry_assets)
    with evaluation_path.open("xb") as file:
        file.write(evaluation_bytes)
    save_registry_settings({
        "verified_assets": registry_assets,
        "evaluation_report": evaluation_artifact,
    })
    print("Evaluation report saved.")

registry_config = project_config["model_registry"]
print("Evaluation split:", registry_evaluation["evaluation_split"])
print("Evaluated tickets:", registry_evaluation["evaluated_tickets"])
for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    print(f"{metric}: {test_metrics[metric]:.4f}")


In [ ]:
import uuid
from botocore.exceptions import ClientError

model_group_name = "aai540-group3-ticket-routing"

registration_request = {
    "ModelPackageGroupName": model_group_name,
    "ModelPackageDescription": (
        "Tested TF-IDF and Logistic Regression ticket-routing model. "
        "Includes the serving code used for successful batch predictions. "
        "Course demonstration; requires review before operational use."
    ),
    "ModelApprovalStatus": "PendingManualApproval",
    "InferenceSpecification": {
        "Containers": [{
            "Image": registry_assets["serving_image"],
            "ModelDataUrl": registry_assets["model_artifact"]["uri"],
            "Environment": registry_assets["serving_environment"],
        }],
        "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
        "SupportedContentTypes": ["application/jsonlines"],
        "SupportedResponseMIMETypes": ["application/jsonlines"],
    },
    "ModelMetrics": {
        "ModelQuality": {
            "Statistics": {
                "ContentType": "application/json",
                "S3Uri": evaluation_artifact["uri"],
            }
        }
    },
    "CustomerMetadataProperties": {
        "training_job": final_model["training_job_name"],
        "dataset_sha256": project_config["dataset_sha256"],
        "model_sha256": registry_assets["model_artifact"]["sha256"],
        "model_s3_version": registry_assets["model_artifact"]["version_id"],
        "inference_code_sha256": registry_assets[
            "inference_code_artifact"
        ]["sha256"],
        "inference_code_s3_version": registry_assets[
            "inference_code_artifact"
        ]["version_id"],
        "evaluation_sha256": evaluation_artifact["sha256"],
        "evaluation_s3_version": evaluation_artifact["version_id"],
        "selection_metric": "validation_macro_f1",
        "test_accuracy": str(test_metrics["accuracy"]),
        "test_macro_f1": str(test_metrics["macro_f1"]),
        "test_weighted_f1": str(test_metrics["weighted_f1"]),
    },
}

# Use the same token to avoid duplicate registration on retries.
registration_request["ClientToken"] = str(uuid.uuid5(
    uuid.NAMESPACE_URL,
    json.dumps(registration_request, sort_keys=True),
))

if request_path.exists():
    require_match(read_json_object(request_path), registration_request, request_path.name)
if registry_checkpoint.get("registration_client_token"):
    require_match(
        registry_checkpoint["registration_client_token"],
        registration_request["ClientToken"], "Registration token",
    )
if registry_checkpoint.get("model_package_group_name"):
    require_match(
        registry_checkpoint["model_package_group_name"], model_group_name, "Registry group"
    )


def check_registered_package(details):
    require_match(details["ModelPackageArn"], model_package_arn, "Model package ARN")
    require_match(details["ModelPackageGroupName"], model_group_name, "Registry group")
    if registry_checkpoint.get("model_package_version") is not None:
        require_match(
            details["ModelPackageVersion"], registry_checkpoint["model_package_version"],
            "Model package version",
        )
    expected = registration_request["InferenceSpecification"]
    actual = details["InferenceSpecification"]
    if len(actual["Containers"]) != 1:
        raise ValueError("Expected one model container.")
    for key, value in expected["Containers"][0].items():
        require_match(actual["Containers"][0].get(key), value, f"Container {key}")
    for key in (
        "SupportedTransformInstanceTypes", "SupportedContentTypes", "SupportedResponseMIMETypes"
    ):
        require_match(sorted(actual.get(key, [])), sorted(expected[key]), key)
    expected_statistics = registration_request["ModelMetrics"]["ModelQuality"]["Statistics"]
    actual_statistics = details["ModelMetrics"]["ModelQuality"]["Statistics"]
    for key, value in expected_statistics.items():
        require_match(actual_statistics.get(key), value, f"Evaluation report {key}")
    for key, value in registration_request["CustomerMetadataProperties"].items():
        require_match(
            details.get("CustomerMetadataProperties", {}).get(key), value,
            f"Model metadata {key}",
        )


current_config()
if registry_checkpoint.get("model_package_arn"):
    model_package_arn = registry_checkpoint["model_package_arn"]
    package_details = sagemaker_client.describe_model_package(
        ModelPackageName=model_package_arn
    )
    check_registered_package(package_details)
    print("Using the saved model version.")
else:
    # Save the request and token before registration so retries use the same values.
    save_new_json(request_path, registration_request)
    save_registry_settings({
        "model_package_group_name": model_group_name,
        "registration_client_token": registration_request["ClientToken"],
    })
    try:
        group_details = sagemaker_client.describe_model_package_group(
            ModelPackageGroupName=model_group_name
        )
    except ClientError as error:
        error_code = error.response["Error"]["Code"]
        message = error.response["Error"]["Message"].lower()
        missing_group = (
            error_code in {"ResourceNotFound", "ResourceNotFoundException"}
            or (
                error_code == "ValidationException"
                and any(text in message for text in [
                    "does not exist", "could not find", "cannot find", "not found"
                ])
            )
        )
        if not missing_group:
            raise
        group_details = sagemaker_client.create_model_package_group(
            ModelPackageGroupName=model_group_name,
            ModelPackageGroupDescription=(
                "Group 3 multilingual ticket routing models for 10 queues."
            ),
        )

    registration = sagemaker_client.create_model_package(**registration_request)
    model_package_arn = registration["ModelPackageArn"]
    save_registry_settings({"model_package_arn": model_package_arn})
    package_details = sagemaker_client.describe_model_package(
        ModelPackageName=model_package_arn
    )
    check_registered_package(package_details)
    registration_progress = {
        "model_package_version": package_details["ModelPackageVersion"],
        "approval_status": package_details["ModelApprovalStatus"],
    }
    # Mark completion after the summary has been saved in the next cell.
    if package_details["ModelPackageStatus"] != "Completed":
        registration_progress["status"] = package_details["ModelPackageStatus"]
    save_registry_settings(registration_progress)

registry_config = project_config["model_registry"]
print("Model version:", package_details["ModelPackageVersion"])
print("Registration status:", package_details["ModelPackageStatus"])
print("Approval status:", package_details["ModelApprovalStatus"])
print("Model package ARN:", model_package_arn)


In [ ]:
check_registered_package(package_details)
if package_details["ModelPackageStatus"] != "Completed":
    raise RuntimeError("Registration is not complete. Run the registration status cell again.")

registered_container = package_details["InferenceSpecification"]["Containers"][0]
registration_summary = {
    "model_package_group": model_group_name,
    "model_package_arn": model_package_arn,
    "model_version": package_details["ModelPackageVersion"],
    "registration_status": package_details["ModelPackageStatus"],
    "approval_status": package_details["ModelApprovalStatus"],
    "serving_image": registered_container["Image"],
    "model_artifact": registry_assets["model_artifact"],
    "inference_code_artifact": registry_assets["inference_code_artifact"],
    "evaluation_report": evaluation_artifact,
    "serving_configuration_verified": True,
}

if summary_path.exists():
    saved_summary = read_json_object(summary_path)
    # Keep the original status snapshot; approval may have changed since that run.
    for key, value in registration_summary.items():
        if key not in {"registration_status", "approval_status"}:
            require_match(saved_summary.get(key), value, f"Registration summary {key}")
    print("Saved registration details match.")
elif completed_resume:
    raise FileNotFoundError("The saved registration summary is missing.")
else:
    current_config()
    save_new_json(summary_path, registration_summary)
    print("Registration summary saved.")

if registry_checkpoint.get("status") != "Completed":
    save_registry_settings({
        "model_package_version": package_details["ModelPackageVersion"],
        "status": package_details["ModelPackageStatus"],
        "approval_status": package_details["ModelApprovalStatus"],
    })

print("Current approval status:", package_details["ModelApprovalStatus"])


We updated the registry code so rerunning it uses the saved model version and evaluation report. If the saved files or AWS details differ, the code stops for us to check them. We tested the changes locally using fake AWS responses. We have not rerun the updated cells in AWS. The original registration reports are still in `reports/model_registry/`.
